# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
from autoloader.main import main

In [0]:
main()

In [0]:
from pyspark.sql import functions as F

  
schema = "code STRING, calling_code STRING, country STRING"

query = (spark.readStream
                    .format("cloudFiles")
                    .option("cloudFiles.format", "json")
                    .schema(schema)
                    .load("/Volumes/landing/country/data/")
                    .withColumn("ingesttime", F.current_timestamp())
)


In [0]:
(
    query
    .writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/landing/country/checkpoint")
    .outputMode("append")
    .option("mergeSchema", True)
    .trigger(availableNow=True)
    .toTable("bronze.default.country_brz")
)

In [0]:
%sql

SELECT * FROM bronze.bookstore.country_brz